In [1]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

# Paths
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations", "set03")
# Load fn attributes subset
val_fn_attr = pd.read_csv("./annot-data/PIE_annot_attrb_val_fn.csv")

# Function: Extract 32-frame sequence per pedestrian
def extract_sequence_from_xml(ped_id, critical_point):
    """
    Return 32-frame occlusion + bbox + attribute info (action, look)
    ending at the given critical_point for a given ped_id.
    """
    try:
        # Determine which video annotation XML file to read
        set_id, video_id, ped_num = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_annt.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None

        # Parse XML file
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Collect all box entries for this pedestrian
        boxes = []
        for box in root.findall(".//box"):
            pid = None
            occ = 0
            attrs = {"action": "__undefined__","look": "__undefined__"}
            for attr in box.findall("attribute"):
                name = attr.attrib.get("name", "").lower()
                value = attr.text.strip() if attr.text else "__undefined__"
                if name == "id":
                    pid = value
                elif name == "occlusion":
                    occ_text = value.lower()
                    occ = 0 if occ_text == "none" else 1 if occ_text == "part" else 2
                elif name in attrs:
                    attrs[name] = value
            if pid == ped_id:
                boxes.append({
                    "frame": int(box.attrib["frame"]),
                    "xtl": float(box.attrib["xtl"]),
                    "ytl": float(box.attrib["ytl"]),
                    "xbr": float(box.attrib["xbr"]),
                    "ybr": float(box.attrib["ybr"]),
                    "occluded": occ,
                    **attrs
                })
        if not boxes:
            return None
        df_boxes = pd.DataFrame(boxes).sort_values("frame").reset_index(drop=True)

        # Get 32 frames before or up to the critical point
        frames_before = df_boxes[df_boxes["frame"] <= critical_point]
        if frames_before.empty:
            return None
        subset = frames_before.tail(32)

        # Pad with first frame if fewer than 32
        if len(subset) < 32:
            subset = pd.concat([
                pd.DataFrame([subset.iloc[0]] * (32 - len(subset))),
                subset
            ], ignore_index=True)

        # Build final dictionary row
        row = {"ped_id": ped_id}
        for i, (_, r) in enumerate(subset.iterrows(), start=1):
            idx = f"{i:02d}"
            row[f"frame_{idx}"] = int(r["frame"])
            row[f"xtl_{idx}"] = int(round(r["xtl"]))
            row[f"ytl_{idx}"] = int(round(r["ytl"]))
            row[f"xbr_{idx}"] = int(round(r["xbr"]))
            row[f"ybr_{idx}"] = int(round(r["ybr"]))
            row[f"occluded_{idx}"] = int(r["occluded"])
            row[f"action_{idx}"] = r["action"]
            # row[f"gesture_{idx}"] = r["gesture"]
            row[f"look_{idx}"] = r["look"]
        return row

    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Extract data for all fn pedestrians
records = []
for _, row in tqdm(val_fn_attr.iterrows(), total=len(val_fn_attr), desc="Extracting fn sequences (32 frames with attributes)"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    seq_data = extract_sequence_from_xml(ped_id, critical_point)
    if seq_data:
        records.append(seq_data)

# Build and clean DataFrame
df_fn_seq_attrb = pd.DataFrame(records)
frame_cols = [c for c in df_fn_seq_attrb.columns if c.startswith("frame_")]
occ_cols = [c for c in df_fn_seq_attrb.columns if c.startswith("occluded_")]
df_fn_seq_attrb[frame_cols + occ_cols] = df_fn_seq_attrb[frame_cols + occ_cols].astype(int)

# Output summary
print(f"\n✅ Extracted fn subset with 32-frame extended data including action and look.")
print(f"📊 Shape: {df_fn_seq_attrb.shape}")
df_fn_seq_attrb.head()

Extracting fn sequences (32 frames with attributes): 100%|██████████| 30/30 [00:16<00:00,  1.86it/s]


✅ Extracted fn subset with 32-frame extended data including action and look.
📊 Shape: (30, 257)


,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,action_31,look_31,frame_32,xtl_32,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32
0,3_2_290,5530,1604,725,1634,805,0,standing,not-looking,5531,...,standing,not-looking,5561,1778,703,1816,807,0,standing,not-looking
1,3_2_302,11793,218,694,260,817,0,walking,not-looking,11794,...,walking,looking,11824,106,675,157,831,0,walking,looking
2,3_2_303,11882,1616,642,1714,900,0,walking,not-looking,11883,...,walking,not-looking,11913,1619,637,1714,922,0,walking,not-looking
3,3_3_327,17550,382,655,433,826,2,walking,not-looking,17551,...,walking,not-looking,17581,271,634,347,855,0,walking,not-looking
4,3_3_326,17526,522,679,566,835,0,standing,not-looking,17527,...,standing,not-looking,17557,316,635,385,880,0,standing,not-looking


In [2]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

# Paths
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations", "set03")
# Load fp attributes subset
val_fp_attr = pd.read_csv("./annot-data/PIE_annot_attrb_val_fp.csv")

# Function: Extract 32-frame sequence per pedestrian
def extract_sequence_from_xml(ped_id, critical_point):
    """
    Return 32-frame occlusion + bbox + attribute info (action, look)
    ending at the given critical_point for a given ped_id.
    """
    try:
        # Determine which video annotation XML file to read
        set_id, video_id, ped_num = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_annt.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None

        # Parse XML file
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Collect all box entries for this pedestrian
        boxes = []
        for box in root.findall(".//box"):
            pid = None
            occ = 0
            attrs = {"action": "__undefined__","look": "__undefined__"}
            for attr in box.findall("attribute"):
                name = attr.attrib.get("name", "").lower()
                value = attr.text.strip() if attr.text else "__undefined__"
                if name == "id":
                    pid = value
                elif name == "occlusion":
                    occ_text = value.lower()
                    occ = 0 if occ_text == "none" else 1 if occ_text == "part" else 2
                elif name in attrs:
                    attrs[name] = value
            if pid == ped_id:
                boxes.append({
                    "frame": int(box.attrib["frame"]),
                    "xtl": float(box.attrib["xtl"]),
                    "ytl": float(box.attrib["ytl"]),
                    "xbr": float(box.attrib["xbr"]),
                    "ybr": float(box.attrib["ybr"]),
                    "occluded": occ,
                    **attrs
                })
        if not boxes:
            return None
        df_boxes = pd.DataFrame(boxes).sort_values("frame").reset_index(drop=True)

        # Get 32 frames before or up to the critical point
        frames_before = df_boxes[df_boxes["frame"] <= critical_point]
        if frames_before.empty:
            return None
        subset = frames_before.tail(32)

        # Pad with first frame if fewer than 32
        if len(subset) < 32:
            subset = pd.concat([
                pd.DataFrame([subset.iloc[0]] * (32 - len(subset))),
                subset
            ], ignore_index=True)

        # Build final dictionary row
        row = {"ped_id": ped_id}
        for i, (_, r) in enumerate(subset.iterrows(), start=1):
            idx = f"{i:02d}"
            row[f"frame_{idx}"] = int(r["frame"])
            row[f"xtl_{idx}"] = int(round(r["xtl"]))
            row[f"ytl_{idx}"] = int(round(r["ytl"]))
            row[f"xbr_{idx}"] = int(round(r["xbr"]))
            row[f"ybr_{idx}"] = int(round(r["ybr"]))
            row[f"occluded_{idx}"] = int(r["occluded"])
            row[f"action_{idx}"] = r["action"]
            # row[f"gesture_{idx}"] = r["gesture"]
            row[f"look_{idx}"] = r["look"]
        return row

    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Extract data for all fp pedestrians
records = []
for _, row in tqdm(val_fp_attr.iterrows(), total=len(val_fp_attr), desc="Extracting fp sequences (32 frames with attributes)"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    seq_data = extract_sequence_from_xml(ped_id, critical_point)
    if seq_data:
        records.append(seq_data)

# Build and clean DataFrame
df_fp_seq_attrb = pd.DataFrame(records)
frame_cols = [c for c in df_fp_seq_attrb.columns if c.startswith("frame_")]
occ_cols = [c for c in df_fp_seq_attrb.columns if c.startswith("occluded_")]
df_fp_seq_attrb[frame_cols + occ_cols] = df_fp_seq_attrb[frame_cols + occ_cols].astype(int)

# Output summary
print(f"\n✅ Extracted fp subset with 32-frame extended data including action and look.")
print(f"📊 Shape: {df_fp_seq_attrb.shape}")
df_fp_seq_attrb.head()

Extracting fp sequences (32 frames with attributes): 100%|██████████| 30/30 [00:16<00:00,  1.86it/s]


✅ Extracted fp subset with 32-frame extended data including action and look.
📊 Shape: (30, 257)


,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,action_31,look_31,frame_32,xtl_32,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32
0,3_1_266,16894,386,697,435,847,0,standing,not-looking,16895,...,standing,not-looking,16925,259,690,331,857,0,standing,not-looking
1,3_1_248,9045,1302,711,1350,854,1,walking,not-looking,9046,...,walking,looking,9076,1575,660,1675,918,1,walking,looking
2,3_2_296,9978,1413,728,1445,855,0,standing,looking,9979,...,standing,looking,10009,1559,703,1598,871,0,standing,looking
3,3_2_295,9979,1437,716,1481,844,2,standing,not-looking,9980,...,standing,not-looking,10010,1579,687,1633,858,2,standing,not-looking
4,3_2_294,9978,1443,717,1481,844,2,standing,not-looking,9979,...,standing,not-looking,10009,1591,692,1647,856,0,standing,not-looking


In [3]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

# Paths
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations", "set03")
# Load tp attributes subset
val_tp_attr = pd.read_csv("./annot-data/PIE_annot_attrb_val_tp.csv")

# Function: Extract 32-frame sequence per pedestrian
def extract_sequence_from_xml(ped_id, critical_point):
    """
    Return 32-frame occlusion + bbox + attribute info (action, look)
    ending at the given critical_point for a given ped_id.
    """
    try:
        # Determine which video annotation XML file to read
        set_id, video_id, ped_num = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_annt.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None

        # Parse XML file
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Collect all box entries for this pedestrian
        boxes = []
        for box in root.findall(".//box"):
            pid = None
            occ = 0
            attrs = {"action": "__undefined__","look": "__undefined__"}
            for attr in box.findall("attribute"):
                name = attr.attrib.get("name", "").lower()
                value = attr.text.strip() if attr.text else "__undefined__"
                if name == "id":
                    pid = value
                elif name == "occlusion":
                    occ_text = value.lower()
                    occ = 0 if occ_text == "none" else 1 if occ_text == "part" else 2
                elif name in attrs:
                    attrs[name] = value
            if pid == ped_id:
                boxes.append({
                    "frame": int(box.attrib["frame"]),
                    "xtl": float(box.attrib["xtl"]),
                    "ytl": float(box.attrib["ytl"]),
                    "xbr": float(box.attrib["xbr"]),
                    "ybr": float(box.attrib["ybr"]),
                    "occluded": occ,
                    **attrs
                })
        if not boxes:
            return None
        df_boxes = pd.DataFrame(boxes).sort_values("frame").reset_index(drop=True)

        # Get 32 frames before or up to the critical point
        frames_before = df_boxes[df_boxes["frame"] <= critical_point]
        if frames_before.empty:
            return None
        subset = frames_before.tail(32)

        # Pad with first frame if fewer than 32
        if len(subset) < 32:
            subset = pd.concat([
                pd.DataFrame([subset.iloc[0]] * (32 - len(subset))),
                subset
            ], ignore_index=True)

        # Build final dictionary row
        row = {"ped_id": ped_id}
        for i, (_, r) in enumerate(subset.iterrows(), start=1):
            idx = f"{i:02d}"
            row[f"frame_{idx}"] = int(r["frame"])
            row[f"xtl_{idx}"] = int(round(r["xtl"]))
            row[f"ytl_{idx}"] = int(round(r["ytl"]))
            row[f"xbr_{idx}"] = int(round(r["xbr"]))
            row[f"ybr_{idx}"] = int(round(r["ybr"]))
            row[f"occluded_{idx}"] = int(r["occluded"])
            row[f"action_{idx}"] = r["action"]
            # row[f"gesture_{idx}"] = r["gesture"]
            row[f"look_{idx}"] = r["look"]
        return row

    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Extract data for all tp pedestrians
records = []
for _, row in tqdm(val_tp_attr.iterrows(), total=len(val_tp_attr), desc="Extracting tp sequences (32 frames with attributes)"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    seq_data = extract_sequence_from_xml(ped_id, critical_point)
    if seq_data:
        records.append(seq_data)

# Build and clean DataFrame
df_tp_seq_attrb = pd.DataFrame(records)
frame_cols = [c for c in df_tp_seq_attrb.columns if c.startswith("frame_")]
occ_cols = [c for c in df_tp_seq_attrb.columns if c.startswith("occluded_")]
df_tp_seq_attrb[frame_cols + occ_cols] = df_tp_seq_attrb[frame_cols + occ_cols].astype(int)

# Output summary
print(f"\n✅ Extracted tp subset with 32-frame extended data including action and look.")
print(f"📊 Shape: {df_tp_seq_attrb.shape}")
df_tp_seq_attrb.head()

Extracting tp sequences (32 frames with attributes): 100%|██████████| 177/177 [02:18<00:00,  1.27it/s]


✅ Extracted tp subset with 32-frame extended data including action and look.
📊 Shape: (177, 257)


,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,action_31,look_31,frame_32,xtl_32,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32
0,3_1_267,16992,502,706,540,837,2,standing,looking,16993,...,standing,looking,17023,477,701,518,844,0,standing,looking
1,3_1_268,17051,397,685,430,817,1,walking,not-looking,17052,...,walking,not-looking,17082,437,689,481,829,0,walking,not-looking
2,3_2_289,5594,507,685,540,820,0,standing,not-looking,5595,...,standing,looking,5625,331,655,378,838,0,standing,looking
3,3_2_304,12582,1790,605,1868,895,0,walking,not-looking,12583,...,walking,not-looking,12613,1615,625,1718,975,0,walking,not-looking
4,3_2_305,12662,1850,608,1918,891,0,walking,not-looking,12663,...,walking,not-looking,12693,1693,627,1800,957,0,walking,not-looking


In [4]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

# Paths
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations", "set03")
# Load tn attributes subset
val_tn_attr = pd.read_csv("./annot-data/PIE_annot_attrb_val_tn.csv")

# Function: Extract 32-frame sequence per pedestrian
def extract_sequence_from_xml(ped_id, critical_point):
    """
    Return 32-frame occlusion + bbox + attribute info (action, look)
    ending at the given critical_point for a given ped_id.
    """
    try:
        # Determine which video annotation XML file to read
        set_id, video_id, ped_num = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_annt.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None

        # Parse XML file
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Collect all box entries for this pedestrian
        boxes = []
        for box in root.findall(".//box"):
            pid = None
            occ = 0
            attrs = {"action": "__undefined__","look": "__undefined__"}
            for attr in box.findall("attribute"):
                name = attr.attrib.get("name", "").lower()
                value = attr.text.strip() if attr.text else "__undefined__"
                if name == "id":
                    pid = value
                elif name == "occlusion":
                    occ_text = value.lower()
                    occ = 0 if occ_text == "none" else 1 if occ_text == "part" else 2
                elif name in attrs:
                    attrs[name] = value
            if pid == ped_id:
                boxes.append({
                    "frame": int(box.attrib["frame"]),
                    "xtl": float(box.attrib["xtl"]),
                    "ytl": float(box.attrib["ytl"]),
                    "xbr": float(box.attrib["xbr"]),
                    "ybr": float(box.attrib["ybr"]),
                    "occluded": occ,
                    **attrs
                })
        if not boxes:
            return None
        df_boxes = pd.DataFrame(boxes).sort_values("frame").reset_index(drop=True)

        # Get 32 frames before or up to the critical point
        frames_before = df_boxes[df_boxes["frame"] <= critical_point]
        if frames_before.empty:
            return None
        subset = frames_before.tail(32)

        # Pad with first frame if fewer than 32
        if len(subset) < 32:
            subset = pd.concat([
                pd.DataFrame([subset.iloc[0]] * (32 - len(subset))),
                subset
            ], ignore_index=True)

        # Build final dictionary row
        row = {"ped_id": ped_id}
        for i, (_, r) in enumerate(subset.iterrows(), start=1):
            idx = f"{i:02d}"
            row[f"frame_{idx}"] = int(r["frame"])
            row[f"xtl_{idx}"] = int(round(r["xtl"]))
            row[f"ytl_{idx}"] = int(round(r["ytl"]))
            row[f"xbr_{idx}"] = int(round(r["xbr"]))
            row[f"ybr_{idx}"] = int(round(r["ybr"]))
            row[f"occluded_{idx}"] = int(r["occluded"])
            row[f"action_{idx}"] = r["action"]
            # row[f"gesture_{idx}"] = r["gesture"]
            row[f"look_{idx}"] = r["look"]
        return row

    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Extract data for all tn pedestrians
records = []
for _, row in tqdm(val_tn_attr.iterrows(), total=len(val_tn_attr), desc="Extracting tn sequences (32 frames with attributes)"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    seq_data = extract_sequence_from_xml(ped_id, critical_point)
    if seq_data:
        records.append(seq_data)

# Build and clean DataFrame
df_tn_seq_attrb = pd.DataFrame(records)
frame_cols = [c for c in df_tn_seq_attrb.columns if c.startswith("frame_")]
occ_cols = [c for c in df_tn_seq_attrb.columns if c.startswith("occluded_")]
df_tn_seq_attrb[frame_cols + occ_cols] = df_tn_seq_attrb[frame_cols + occ_cols].astype(int)

# Output summary
print(f"\n✅ Extracted tn subset with 32-frame extended data including action and look.")
print(f"📊 Shape: {df_tn_seq_attrb.shape}")
df_tn_seq_attrb.head()

Extracting tn sequences (32 frames with attributes): 100%|██████████| 482/482 [04:30<00:00,  1.78it/s]


✅ Extracted tn subset with 32-frame extended data including action and look.
📊 Shape: (482, 257)


,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,action_31,look_31,frame_32,xtl_32,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32
0,3_1_232,3403,481,723,508,796,0,walking,not-looking,3404,...,standing,not-looking,3434,252,699,289,811,0,standing,not-looking
1,3_1_237,7207,1371,704,1408,840,0,standing,not-looking,7208,...,standing,not-looking,7238,1525,672,1574,862,0,standing,not-looking
2,3_1_255,9144,419,722,450,820,0,standing,not-looking,9145,...,standing,not-looking,9175,92,673,135,833,0,standing,not-looking
3,3_1_231,3229,554,729,577,805,0,standing,looking,3230,...,standing,not-looking,3260,289,699,324,824,0,standing,not-looking
4,3_1_224,2777,469,732,490,815,0,standing,not-looking,2778,...,standing,not-looking,2808,246,719,276,842,0,standing,not-looking


In [5]:
# Save FN, FP, TP, TN sets
df_fn_seq_attrb.to_csv('./occ-data/PIE_attrb_fn_ext.csv', index=0)
df_fp_seq_attrb.to_csv('./occ-data/PIE_attrb_fp_ext.csv', index=0)
df_tp_seq_attrb.to_csv('./occ-data/PIE_attrb_tp_ext.csv', index=0)
df_tn_seq_attrb.to_csv('./occ-data/PIE_attrb_tn_ext.csv', index=0)

In [7]:
val_obd = pd.read_csv('./PIE_val_vehicle.csv')
val_obd.head()

,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state,subset
0,3_2_290,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,...,20.004146,20.004146,20.004146,18.008559,18.008559,18.008559,18.008559,18.008559,constant,FN
1,3_2_302,15.996879,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,...,13.003500,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,decelerating,FN
2,3_2_303,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,decelerating,FN
3,3_3_309,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,...,28.002586,28.002586,27.004792,26.006999,26.006999,26.006999,26.006999,26.006999,constant,FN
4,3_3_326,19.006353,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,...,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,decelerating,FN


In [8]:
val_attr = pd.read_csv('./PIE_annot_attrb_val.csv')
val_attr.head()

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,3434,0,3452,3361,male,3_1_232,0.950000,four-way,3,CS,OW,video_0001_attributes.xml
1,adult,7238,0,7275,7148,male,3_1_237,0.950000,four-way,4,CS,OW,video_0001_attributes.xml
2,adult,9175,0,9180,9141,female,3_1_255,0.916667,four-way,3,CS,OW,video_0001_attributes.xml
3,adult,17023,1,17029,16933,male,3_1_267,0.833333,midblock,3,NaN,OW,video_0001_attributes.xml
4,adult,3260,0,3278,3170,male,3_1_231,0.766667,midblock,3,NaN,OW,video_0001_attributes.xml


In [9]:
occ_fn = df_fn_seq_attrb.copy()
occ_fp = df_fp_seq_attrb.copy()
occ_tp = df_tp_seq_attrb.copy()
occ_tn = df_tn_seq_attrb.copy()
occ_tn.head()

,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,action_31,look_31,frame_32,xtl_32,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32
0,3_1_232,3403,481,723,508,796,0,walking,not-looking,3404,...,standing,not-looking,3434,252,699,289,811,0,standing,not-looking
1,3_1_237,7207,1371,704,1408,840,0,standing,not-looking,7208,...,standing,not-looking,7238,1525,672,1574,862,0,standing,not-looking
2,3_1_255,9144,419,722,450,820,0,standing,not-looking,9145,...,standing,not-looking,9175,92,673,135,833,0,standing,not-looking
3,3_1_231,3229,554,729,577,805,0,standing,looking,3230,...,standing,not-looking,3260,289,699,324,824,0,standing,not-looking
4,3_1_224,2777,469,732,490,815,0,standing,not-looking,2778,...,standing,not-looking,2808,246,719,276,842,0,standing,not-looking


In [10]:
import pandas as pd
action_cols   = [col for col in occ_tn.columns if col.startswith('action_')]
look_cols     = [col for col in occ_tn.columns if col.startswith('look_')]
occluded_cols = [col for col in occ_tn.columns if col.startswith('occluded_')]

def dominant_action(row):
    counts = row.value_counts()
    walking_count  = counts.get('walking', 0)
    standing_count = counts.get('standing', 0)
    return 'walking' if walking_count >= standing_count else 'standing'

def dominant_look(row):
    counts = row.value_counts()
    looking_count     = counts.get('looking', 0)
    not_looking_count = counts.get('not-looking', 0)
    return 'looking' if looking_count >= not_looking_count else 'not-looking'

def dominant_occlusion(row):
    if (row == 2).any():     return 'occluded'
    elif (row == 1).any():   return 'partially-occluded'
    else:                    return 'not-occluded'

occ_tn['behaviour_action'] = occ_tn[action_cols].apply(dominant_action, axis=1)
occ_tn['behaviour_look']   = occ_tn[look_cols].apply(dominant_look, axis=1)
occ_tn['occlusion_state']  = occ_tn[occluded_cols].apply(dominant_occlusion, axis=1)
occ_tn['behaviour_state']  = occ_tn['behaviour_action'] + ' + ' + occ_tn['behaviour_look']

occ_tn.head()

,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,ytl_32,xbr_32,ybr_32,occluded_32,action_32,look_32,behaviour_action,behaviour_look,occlusion_state,behaviour_state
0,3_1_232,3403,481,723,508,796,0,walking,not-looking,3404,...,699,289,811,0,standing,not-looking,walking,not-looking,not-occluded,walking + not-looking
1,3_1_237,7207,1371,704,1408,840,0,standing,not-looking,7208,...,672,1574,862,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking
2,3_1_255,9144,419,722,450,820,0,standing,not-looking,9145,...,673,135,833,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking
3,3_1_231,3229,554,729,577,805,0,standing,looking,3230,...,699,324,824,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking
4,3_1_224,2777,469,732,490,815,0,standing,not-looking,2778,...,719,276,842,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking


In [11]:
occ_tp['behaviour_action'] = occ_tp[action_cols].apply(dominant_action, axis=1)
occ_tp['behaviour_look']   = occ_tp[look_cols].apply(dominant_look, axis=1)
occ_tp['occlusion_state']  = occ_tp[occluded_cols].apply(dominant_occlusion, axis=1)
occ_tp['behaviour_state']  = occ_tp['behaviour_action'] + ' + ' + occ_tp['behaviour_look']

occ_fn['behaviour_action'] = occ_fn[action_cols].apply(dominant_action, axis=1)
occ_fn['behaviour_look']   = occ_fn[look_cols].apply(dominant_look, axis=1)
occ_fn['occlusion_state']  = occ_fn[occluded_cols].apply(dominant_occlusion, axis=1)
occ_fn['behaviour_state']  = occ_fn['behaviour_action'] + ' + ' + occ_fn['behaviour_look']

occ_fp['behaviour_action'] = occ_fp[action_cols].apply(dominant_action, axis=1)
occ_fp['behaviour_look']   = occ_fp[look_cols].apply(dominant_look, axis=1)
occ_fp['occlusion_state']  = occ_fp[occluded_cols].apply(dominant_occlusion, axis=1)
occ_fp['behaviour_state']  = occ_fp['behaviour_action'] + ' + ' + occ_fp['behaviour_look']

In [12]:
occ_fn_set = occ_fn[['ped_id', 'occlusion_state', 'behaviour_state']]
occ_fp_set = occ_fp[['ped_id', 'occlusion_state', 'behaviour_state']]
occ_tp_set = occ_tp[['ped_id', 'occlusion_state', 'behaviour_state']]
occ_tn_set = occ_tn[['ped_id', 'occlusion_state', 'behaviour_state']]

In [13]:
val_full = pd.concat([occ_fn_set, occ_fp_set, occ_tp_set, occ_tn_set], ignore_index=True)
print(val_full.shape)
val_full.head()

(719, 3)


,ped_id,occlusion_state,behaviour_state
0,3_2_290,not-occluded,standing + not-looking
1,3_2_302,not-occluded,walking + looking
2,3_2_303,not-occluded,walking + not-looking
3,3_3_327,occluded,walking + not-looking
4,3_3_326,not-occluded,standing + not-looking


In [14]:
def get_intersection(row):
    inter = val_attr[val_attr['id'] == row['ped_id']]['intersection'].values[0]
    if inter == 'four-way': return inter
    elif inter == 'midblock': return inter
    else: return 'T-junction'

def get_motion_state(row):
    motion = val_obd[val_obd['ped_id'] == row['ped_id']]['motion_state'].values[0]
    return motion

# def get_crossing(row):
#     crossing = val_attr[val_attr['id'] == row['ped_id']]['crossing'].values[0]
#     if crossing == 1: return crossing
#     else: return 0

def get_subset(row):
    subset = val_obd[val_obd['ped_id'] == row['ped_id']]['subset'].values[0]
    return subset

In [15]:
val_full['intersection_type'] = val_full.apply(get_intersection, axis=1)
val_full['motion_state'] = val_full.apply(get_motion_state, axis=1)
# val_full['crossing_event'] = val_full.apply(get_crossing, axis=1)
val_full['prediction_subset'] = val_full.apply(get_subset, axis=1)
val_full.head()

,ped_id,occlusion_state,behaviour_state,intersection_type,motion_state,prediction_subset
0,3_2_290,not-occluded,standing + not-looking,four-way,constant,FN
1,3_2_302,not-occluded,walking + looking,four-way,decelerating,FN
2,3_2_303,not-occluded,walking + not-looking,four-way,decelerating,FN
3,3_3_327,occluded,walking + not-looking,T-junction,decelerating,FN
4,3_3_326,not-occluded,standing + not-looking,T-junction,decelerating,FN


In [16]:
val_all_updated =  val_full.replace('looking', 'facing', regex=True)
val_all_updated.head()

,ped_id,occlusion_state,behaviour_state,intersection_type,motion_state,prediction_subset
0,3_2_290,not-occluded,standing + not-facing,four-way,constant,FN
1,3_2_302,not-occluded,walking + facing,four-way,decelerating,FN
2,3_2_303,not-occluded,walking + not-facing,four-way,decelerating,FN
3,3_3_327,occluded,walking + not-facing,T-junction,decelerating,FN
4,3_3_326,not-occluded,standing + not-facing,T-junction,decelerating,FN


In [17]:
occ_full = pd.concat([occ_fn, occ_fp, occ_tp, occ_tn], ignore_index=True)
occ_full['subset'] = val_all_updated['prediction_subset']
occ_full.head()

,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,xbr_32,ybr_32,occluded_32,action_32,look_32,behaviour_action,behaviour_look,occlusion_state,behaviour_state,subset
0,3_2_290,5530,1604,725,1634,805,0,standing,not-looking,5531,...,1816,807,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking,FN
1,3_2_302,11793,218,694,260,817,0,walking,not-looking,11794,...,157,831,0,walking,looking,walking,looking,not-occluded,walking + looking,FN
2,3_2_303,11882,1616,642,1714,900,0,walking,not-looking,11883,...,1714,922,0,walking,not-looking,walking,not-looking,not-occluded,walking + not-looking,FN
3,3_3_327,17550,382,655,433,826,2,walking,not-looking,17551,...,347,855,0,walking,not-looking,walking,not-looking,occluded,walking + not-looking,FN
4,3_3_326,17526,522,679,566,835,0,standing,not-looking,17527,...,385,880,0,standing,not-looking,standing,not-looking,not-occluded,standing + not-looking,FN


In [18]:
val_occ_updated = occ_full.replace('looking', 'facing', regex=True)
val_occ_updated.head()

,ped_id,frame_01,xtl_01,ytl_01,xbr_01,ybr_01,occluded_01,action_01,look_01,frame_02,...,xbr_32,ybr_32,occluded_32,action_32,look_32,behaviour_action,behaviour_look,occlusion_state,behaviour_state,subset
0,3_2_290,5530,1604,725,1634,805,0,standing,not-facing,5531,...,1816,807,0,standing,not-facing,standing,not-facing,not-occluded,standing + not-facing,FN
1,3_2_302,11793,218,694,260,817,0,walking,not-facing,11794,...,157,831,0,walking,facing,walking,facing,not-occluded,walking + facing,FN
2,3_2_303,11882,1616,642,1714,900,0,walking,not-facing,11883,...,1714,922,0,walking,not-facing,walking,not-facing,not-occluded,walking + not-facing,FN
3,3_3_327,17550,382,655,433,826,2,walking,not-facing,17551,...,347,855,0,walking,not-facing,walking,not-facing,occluded,walking + not-facing,FN
4,3_3_326,17526,522,679,566,835,0,standing,not-facing,17527,...,385,880,0,standing,not-facing,standing,not-facing,not-occluded,standing + not-facing,FN


In [19]:
val_occ_updated.to_csv('./PIE_val_occ_updated.csv', index=0)
val_all_updated.to_csv('./PIE_val_all_updated.csv', index=0)